To be converted to scripts eventually?

TODO:
get additional metadata from https://www.agathachristie.com/
- orig pub date
- work type
- detective
- alt titles?

Desired columns for LIB

- publisher ?
- title (from "title") 
- subject 1, subject 2, subject 3, (or more?), 
- series from "belongs-to-collection"?
- detective (add later) 
- collection type from "collection type"
- series position from "group-position"
- short description from "description"
- long description from "long-description"
- language from dc:language
- se word-count from "se:word-count"
- flesch scale from "se:reading-ease.flesch"
- source_url from "se:url.vcs.github"
- author from "author"
- author full name? from "se:name.person.full-name" (DON'T NEED)
- wikipedia url from "se:url.encyclopedia.wikipedia"
- number of chapters (DO LATER IN CORPUS)
- written_as (manually)
- work_type (manually)
- alt_titles (LATER) (manually)

Design decisions:
- choosing to treat each story in poirot investigates as its own document since the stories are the unit of composition (4/9/26)

## Raw Data Ingestion

The script below was run once to clone the corpus repositories from Standard Ebooks. 
It is preserved here for reproducibility. Do not re-run — raw_data/ is already populated.

```bash
#!/bin/bash
# Ingestion script for Agatha Christie Standard Ebooks Corpus

mkdir raw_data
cd raw_data

echo "Cloning repositories..."
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-at-the-vicarage.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_giants-bread.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-secret-adversary.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-on-the-links.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-man-in-the-brown-suit.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-big-four.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_poirot-investigates.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-mysterious-affair-at-styles.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-secret-of-chimneys.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-seven-dials-mystery.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-murder-of-roger-ackroyd.git
git clone --depth 1 git@github.com:standardebooks/agatha-christie_the-mystery-of-the-blue-train.git

echo "Ingestion complete."
```

## Construct LIB table

In [ ]:
# import libraries
import pandas as pd
import glob
import os
import pathlib
from bs4 import BeautifulSoup
import re

In [4]:
# get list of all files in the raw_data folder
root_dir = 'raw_data'
file_list = []

for root, dirs, files in os.walk(root_dir):
    for file in files:
        # Join the folder path and file name to get the full path
        full_path = os.path.join(root, file)
        file_list.append(full_path)

# file_list

In [ ]:
# get paths to content.opfs in the raw_data folder

root_dir = 'raw_data'
path_list = []


for root, dirs in os.walk(root_dir):
    for dir in dirs:
        # Join the folder path and file name to get the full path
        full_path = os.path.join(root, dir)
        path_list.append(full_path)


In [15]:
print(LIB.to_string())
print("\nMissing values:")
print(LIB.isnull().sum())

                                                  book_id                            title           author               se_date  pub_year word_count flesch_ease language                                                                                                                                          description                                                                                                                                                                                                                                                  subjects                se_subjects                 series series_pos                                                  wikipedia_url                                                                             se_url  n_chapters
book_num                                                                                                                                                                                                          

In [20]:
# fix pub year missingness by hand (from agathachristie.com):

missing_pub_years = {
    'The Big Four': 1927, # confirmed
    'The Murder at the Vicarage': 1930, # confirmed
    'The Murder of Roger Ackroyd': 1926, # confirmed
    'The Murder on the Links': 1923, # confirmed
    'The Mysterious Affair at Styles': 1920, # confirmed
    'The Mystery of the Blue Train': 1928, # confirmed
    'The Secret Adversary': 1922, # confirmed
    'The Secret of Chimneys': 1925, # confirmed
    'The Seven Dials Mystery': 1929, # confirmed
}

LIB['pub_year'] = LIB.apply(
    lambda row: missing_pub_years.get(row['title'], row['pub_year']),
    axis=1
).astype(int)

LIB.to_csv('LIB.csv', sep='|', index=True)
print(LIB.to_string())

                                                  book_id                            title           author               se_date  pub_year word_count flesch_ease language                                                                                                                                          description                                                                                                                                                                                                                                                  subjects                se_subjects                 series series_pos                                                  wikipedia_url                                                                             se_url  n_chapters
book_num                                                                                                                                                                                                          

In [30]:
dict = {}
for i in range(12):
    dict[LIB['title'].iloc[i]] = 'novel'
print(dict)

{'Giant’s Bread': 'novel', 'Poirot Investigates': 'novel', 'The Big Four': 'novel', 'The Man in the Brown Suit': 'novel', 'The Murder at the Vicarage': 'novel', 'The Murder of Roger Ackroyd': 'novel', 'The Murder on the Links': 'novel', 'The Mysterious Affair at Styles': 'novel', 'The Mystery of the Blue Train': 'novel', 'The Secret Adversary': 'novel', 'The Secret of Chimneys': 'novel', 'The Seven Dials Mystery': 'novel'}


In [ ]:
# add work type and writing_as by hand (from agathachristie.com):
work_types = {
    'Giant’s Bread': 'novel',
    'Poirot Investigates': 'novel',
    'The Big Four': 'novel',
    'The Man in the Brown Suit': 'novel',
    'The Murder at the Vicarage': 'novel',
    'The Murder of Roger Ackroyd': 'novel',
    'The Murder on the Links': 'novel',
    'The Mysterious Affair at Styles': 'novel',
    'The Mystery of the Blue Train': 'novel',
    'The Secret Adversary': 'novel',
    'The Secret of Chimneys': 'novel',
    'The Seven Dials Mystery': 'novel'
}

writing_as = {
    'Giant’s Bread': 'novel',
    'Poirot Investigates': 'novel',
    'The Big Four': 'novel',
    'The Man in the Brown Suit': 'novel',
    'The Murder at the Vicarage': 'novel',
    'The Murder of Roger Ackroyd': 'novel',
    'The Murder on the Links': 'novel',
    'The Mysterious Affair at Styles': 'novel',
    'The Mystery of the Blue Train': 'novel',
    'The Secret Adversary': 'novel',
    'The Secret of Chimneys': 'novel',
    'The Seven Dials Mystery': 'novel'
}

LIB['work_type'] = LIB.apply(
    lambda row: work_types.get(row['title'], row['work_type']),
    axis=1
)

LIB['writing_as'] = LIB.apply(
    lambda row: writing_as.get(row['title'], row['writing_as']),
    axis=1
).astype(int)

LIB.to_csv('LIB.csv', sep='|', index=True)
print(LIB.to_string())